CONTENT_BASED

Ce code construit un petit système de recommandation basé sur les genres. Il commence par définir un transformer GenresBinarizer qui convertit la colonne genres_list (liste de genres par film) en colonnes numériques 0/1 utilisables par un modèle. Ensuite, il crée un pipeline scikit‑learn: d’abord un prétraitement (ColumnTransformer) qui applique GenresBinarizer aux genres et laisse passer des features agrégées (user_mean_rating, movie_mean_rating, movie_rating_count), puis un DecisionTreeClassifier qui apprend, à partir de l’historique, si un utilisateur aime (1) ou non (0) un film. Après avoir séparé les données en train/test et entraîné le pipeline, on évalue l’accuracy. Enfin, on construit une table movies avec un film par ligne et on définit recommend_for_user, qui pour un utilisateur donné supprime les films déjà vus, calcule la probabilité qu’il aime chaque film restant via le pipeline, et renvoie les films non vus avec la plus forte probabilité d’être aimés, en tenant compte aussi de leur note moyenne globale.

Importation de CSV

In [32]:
import pandas as pd
import matplotlib.pyplot as plt

ratings = pd.read_csv('ratings.csv')
movies = pd.read_csv('movies.csv')

df = ratings.merge(movies, on='movieId')

df.head()

,userId,movieId,rating,timestamp,title,genres
0,1,16,4.0,1217897793,Casino (1995),Crime|Drama
1,1,24,1.5,1217895807,Powder (1995),Drama|Sci-Fi
2,1,32,4.0,1217896246,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Mystery|Sci-Fi|Thriller
3,1,47,4.0,1217896556,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,4.0,1217896523,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [33]:
df.isna().sum()

userId       0
movieId      0
rating       0
timestamp    0
title        0
genres       0
dtype: int64

Separation de donnees

In [34]:
df["genres_list"] = df["genres"].str.split("|")

Mettre en temps

In [35]:
df['like'] = (df["rating"] >= 3).astype(int)

In [36]:
df

,userId,movieId,rating,timestamp,title,genres,genres_list,like
0,1,16,4.0,1217897793,Casino (1995),Crime|Drama,"[Crime, Drama]",1
1,1,24,1.5,1217895807,Powder (1995),Drama|Sci-Fi,"[Drama, Sci-Fi]",0
2,1,32,4.0,1217896246,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Mystery|Sci-Fi|Thriller,"[Mystery, Sci-Fi, Thriller]",1
3,1,47,4.0,1217896556,Seven (a.k.a. Se7en) (1995),Mystery|Thriller,"[Mystery, Thriller]",1
4,1,50,4.0,1217896523,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,"[Crime, Mystery, Thriller]",1
...,...,...,...,...,...,...,...,...
105334,668,142488,4.0,1451535844,Spotlight (2015),Thriller,[Thriller],1
105335,668,142507,3.5,1451535889,Pawn Sacrifice (2015),Drama,[Drama],1
105336,668,143385,4.0,1446388585,Bridge of Spies (2015),Drama|Thriller,"[Drama, Thriller]",1
105337,668,144976,2.5,1448656898,Bone Tomahawk (2015),Horror|Western,"[Horror, Western]",0


Creation de colonne pour predire si le user aime ou non un film.

In [37]:
user_stats = df.groupby("userId")["rating"].agg(user_mean_rating="mean")
movie_stats = df.groupby("movieId")["rating"].agg(
    movie_mean_rating="mean",
    movie_rating_count="count"
)

df = df.merge(user_stats, on="userId", how="left")
df = df.merge(movie_stats, on="movieId", how="left")

Si utilisateur aime ou pas un film

In [38]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
import numpy as np

# Transformer pour les genres
class GenresBinarizer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.mlb = MultiLabelBinarizer()
    def fit(self, X, y=None):
        self.mlb.fit(X.iloc[:, 0])
        return self
    def transform(self, X):
        return self.mlb.transform(X.iloc[:, 0])

# Définir X et y (on ne prend PAS timestamp, title, userId, movieId ici)
feature_cols = ["genres_list", "user_mean_rating", "movie_mean_rating", "movie_rating_count"]
X = df[feature_cols]
y = df["like"]

preprocess = ColumnTransformer(
    transformers=[
        ("genres", GenresBinarizer(), ["genres_list"]),
        ("num", "passthrough", ["user_mean_rating", "movie_mean_rating", "movie_rating_count"])
    ]
)

clf = DecisionTreeClassifier(
    max_depth=10,
    min_samples_leaf=5,
    random_state=42
)

pipe = Pipeline([
    ("preprocess", preprocess),
    ("model", clf)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipe.fit(X_train, y_train)
print("Accuracy:", pipe.score(X_test, y_test))

Accuracy: 0.8567970381621416


utilise model generer ;iste recommander pour un utilisateur donné, triés par probabilité d’aimer et qualité moyenne.

In [39]:
# movies uniques + stats (tu peux réutiliser movie_stats si tu veux)
movies = df.groupby("movieId", as_index=False).agg(
    title=("title", "first"),
    genres_list=("genres_list", "first"),
    movie_mean_rating=("rating", "mean"),
    movie_rating_count=("rating", "count")
)

def recommend_for_user(user_id, df_ratings, movies, pipe, top_n=10):
    # ratings du user
    user_ratings = df_ratings[df_ratings["userId"] == user_id]

    # films déjà vus
    seen = set(user_ratings["movieId"])

    # films candidats
    candidates = movies[~movies["movieId"].isin(seen)].copy()
    if candidates.empty:
        return candidates

    # récupérer la moyenne de rating de ce user (calculée avant)
    user_mean = df_ratings.loc[df_ratings["userId"] == user_id, "user_mean_rating"].iloc[0]

    # construire X_cand avec les mêmes features que pour l'entraînement
    X_cand = candidates.copy()
    X_cand["user_mean_rating"] = user_mean

    # on a déjà movie_mean_rating et movie_rating_count dans `candidates`
    feature_cols = ["genres_list", "user_mean_rating", "movie_mean_rating", "movie_rating_count"]
    X_cand = X_cand[feature_cols]

    proba = pipe.predict_proba(X_cand)[:, 1]
    candidates["like_proba"] = proba

    recs = candidates.sort_values(
        ["like_proba", "movie_mean_rating"],
        ascending=False
    ).head(top_n)

    return recs[["movieId", "title", "like_proba", "movie_mean_rating", "movie_rating_count"]]



Le Test de model

In [44]:
print("\nRecommandations pour l'utilisateur 1:")
print(recommend_for_user(2, df, movies, pipe))


Recommandations pour l'utilisateur 1:
      movieId                                           title  like_proba  \
110       124  Star Maker, The (Uomo delle stelle, L') (1995)         1.0   
197       226                                Dream Man (1995)         1.0   
366       418                              Being Human (1993)         1.0   
412       465                           Heaven & Earth (1993)         1.0   
506       567                                     Kika (1993)         1.0   
517       583                 Dear Diary (Caro Diario) (1994)         1.0   
565       649              Cold Fever (Á köldum klaka) (1995)         1.0   
608       716                      Switchblade Sisters (1975)         1.0   
675       831                                Stonewall (1995)         1.0   
1190     1471                              Boys Life 2 (1997)         1.0   

      movie_mean_rating  movie_rating_count  
110                 5.0                   1  
197                 5